# Model V2 - Feature-Enhanced Model

In [ ]:
import pandas as pd

In [ ]:
# load the engineered feature CSV (produced by feature_engineering.ipynb)
df = pd.read_csv('../data/convertData/finland_electricity_features_v2.csv')
df

### Create train & test data set

In [ ]:
# drop the target variable and the datetime column from the feature set
X = df.drop(columns = ['price', 'datetime'])

# set the dependent variable (what we want to predict = price)
y = df['price']

In [ ]:
from sklearn.model_selection import train_test_split

# spliting the data into training and testing sets (80/20, no shuffle - time series!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, shuffle=False)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
# print the list of feature column names (already clean from feature_engineering.ipynb)
print('Feature columns used in V2:', X_train.columns.tolist())

### TRAIN V2 XGBOOST REGRESSION MODEL

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

In [ ]:
# instantiate the model with the same hyperparameters as V1
model_v2 = XGBRegressor(
    objective='reg:squarederror',
    learning_rate=0.1,
    n_estimators=100,
    max_depth=6,
    random_state=42)

# train the model on the training data
model_v2.fit(X_train, y_train)

In [ ]:
# test the model on the test set and get predictions
y_pred = model_v2.predict(X_test)

# k is the number of features in the test set
k = X_test.shape[1]
# n is the number of samples in the test set
n = len(X_test)

# calculate mse, rmse, mae, r2 score and adjusted r2 score
mse_score = mean_squared_error(y_true=y_test, y_pred=y_pred)

rmse_score = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_pred))

mae_score = mean_absolute_error(y_true=y_test, y_pred=y_pred)

r2_score_val = r2_score(y_true=y_test, y_pred=y_pred)

adjusted_r2 = 1 - (1 - r2_score_val) * (n - 1) / (n - k - 1)

# print the evaluation results
print('================ V2 XGBoost Evaluation ================')
print('Mean Absolute Error (MAE)            :', mae_score)
print('Mean Squared Error (MSE)             :', mse_score)
print('Root Mean Squared Error (RMSE)       :', rmse_score)
print('R2 Score (R2)                        :', r2_score_val)
print('Adjusted R2                           :', adjusted_r2)
print('=======================================================')

# Model V2 Performance Report
## Summary
This model (V2) uses the full engineered feature table (lag features, rolling stats, calendar features, holiday flags, etc.) produced by feature_engineering.ipynb.

## Evaluation Metrics
The model was tested on the final 20% of the time series (chronological split, no shuffle).